In [12]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import torch
import gymnasium as gym
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import DistributionLoss
from neuralforecast.models import TimesNet
from stable_baselines3 import A2C, PPO
from gymnasium.utils.env_checker import check_env

from config.config import appConfig
from src.data_prep import data_prep, data_for_timesnet
from src.portfolio_env import PortfolioEnv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
config = appConfig(
    initial_balance=100000,
    allow_short_selling=False,
    technical_indicator_list=['MACD','RSI_14','CCI_20','SMA_20','EMA_20'], #'ADX_14',
    transaction_cost=0.001,
    ticker_list=["BTC-USD","ETH-USD","LTC-USD","LINK-USD","BCH-USD","UNI-USD","XLM-USD","FIL-USD","BNB-USD","SOL-USD","XRP-USD","ADA-USD","SHIB-USD","TON-USD","DOGE-USD","AVAX-USD","TRX-USD","DOT-USD","MATIC-USD","ETC-USD"],
    train_starting_date="2020-09-22",
    train_ending_date="2022-06-07",
    test_starting_date="2022-06-08",
    test_ending_date="2023-01-03",
    THRESHOLD_PARAMETER=0.015,
    oc_upper_threshold=0.01, oc_lower_threshold=-0.01, oc_k_loss=0.1, oc_k_gain=0.1, oc_n=0.725,
    ra_upper_threshold=0.01, ra_lower_threshold=-0.01, ra_k_loss=0.1, ra_k_gain=0.1, ra_n=1.22,
)


In [18]:
train_df, train_nan_dates = data_prep(config.train_starting_date, config.train_ending_date, config.ticker_list)
test_df, test_nan_dates  = data_prep(config.test_starting_date,  config.test_ending_date,  config.ticker_list)
print(f"Train shape: {train_df.shape}, dates: {train_df['ds'].nunique()}, tickers: {train_df['unique_id'].nunique()}")
print(f"Test  shape: {test_df.shape},  dates: {test_df['ds'].nunique()},  tickers: {test_df['unique_id'].nunique()}")


[*********************100%***********************]  20 of 20 completed
/mnt/c/Users/akgup/Documents/Obsidian Backup/Vault/Files/DL and NLP in Finance/Paper Implementation BBAPT/BBAPT/src/data_prep.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level = 'Ticker').reset_index()
[*********************100%***********************]  20 of 20 completed


Train shape: (7960, 14), dates: 398, tickers: 20
Test  shape: (3680, 14),  dates: 184,  tickers: 20


/mnt/c/Users/akgup/Documents/Obsidian Backup/Vault/Files/DL and NLP in Finance/Paper Implementation BBAPT/BBAPT/src/data_prep.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level = 'Ticker').reset_index()


In [5]:
model = TimesNet(h=1,
                input_size=24, #Context Window
                hidden_size = 16,
                conv_hidden_size = 32,
                loss=DistributionLoss(distribution='Normal', level=[80, 90]),
                scaler_type='standard',
                learning_rate=1e-3,
                max_steps=100,
                val_check_steps=50,
                early_stop_patience_steps=2)
nf = NeuralForecast(
    models=[model],
    freq='B'
)
nf.fit(df=data_for_timesnet(train_df), val_size=1)
forecast = nf.predict()

Seed set to 1


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
forecast.head()

In [62]:
gym.register(
    id='PortfolioEnv-v4',
    entry_point=PortfolioEnv,
    kwargs={'data': train_df, 'config': config}
)
env = gym.make('PortfolioEnv-v4')
try:
    check_env(env.unwrapped)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")


Environment has issues: The first element returned by `env.reset()` is not within the observation space.


/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/gymnasium/envs/registration.py:637: UserWarning: WARN: Overriding environment PortfolioEnv-v4 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/gymnasium/utils/env_checker.py:317: UserWarning: WARN: A Box observation space minimum value is -infinity. This is probably too low.
  logger.warn(
/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/gymnasium/utils/env_checker.py:321: UserWarning: WARN: A Box observation space maximum value is infinity. This is probably too high.
  logger.warn(


In [ ]:
model_rl = A2C('MlpPolicy', env, verbose=1)
model_rl.learn(total_timesteps=10000)
print("Training completed!")

Box(-inf, inf, (20, 7), float32)

In [ ]:
list_forecasts = []
for ticker in config.ticker_list:
    nf.fit(df=data_for_timesnet(train_df.loc[train_df['unique_id'] == ticker]), val_size=1)
    list_forecasts.append(nf.predict())
forecast = pd.concat(list_forecasts,ignore_index=True)
forecast['avg_return'] = forecast.groupby('ds')['TimesNet'].transform('mean')